# Flow Matching vs. DDPM — Inference Steps Study

Compares sample quality as a function of **number of function evaluations (NFE)** for:
- **FlowMatching** (Euler ODE): 5, 10, 25, 50, 100 Euler steps
- **DiffusionSR** (DDIM): skip=50→20 NFE, 20→50, 10→100, 5→200, 1→1000

Uses already-trained models from the main multifield experiment (nb08 / nb09). 
No new training required — run this once checkpoints exist.

Metrics: field MAE, MP-MAE (melt-pool depth error), inference time per sample.  
Output: quality-vs-NFE curves + quality-vs-wall-clock Pareto scatter.


In [ ]:
# ── USER CONFIG ────────────────────────────────────────────────────────────────
# FlowMatching model (from nb09 / main multifield experiment)
FM_RUN_DIR  = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/direct/flowmatchingimplicitencoded/cs_both_n3'
FM_ENC_DIR  = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/direct/encoder/cs_both_n3'
FM_N_STEPS  = 100   # default Euler steps at full quality (matches nb09)

# DiffusionSR model (from nb08 / main multifield experiment)
DIFF_RUN_DIR = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/direct/diffusionimplicitencoded/cs_both_n3'
DIFF_ENC_DIR = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/direct/encoder/cs_both_n3'

# Alternatively, swap in ablation_20260901 enc+sdf runs once trained:
# FM_RUN_DIR  = '/scratch/ngng/runs/abl_fm_enc_sdf'
# FM_ENC_DIR  = '/trace/.../runs/ablation_20260901/enc_sdf'
# DIFF_RUN_DIR = '/scratch/ngng/runs/abl_ldm_enc_sdf'  (LDM — set MODEL2='ldm' below)

# Dataset config — must match the checkpoint's training config
FIELD_NAMES      = ['temperature', 'liqlabel']  # nb08/09 use liqlabel; change if using sdf runs
N_STEPS          = 3
DOWNSCALE_METHOD = 'direct'
NORMALIZE        = 'standardize'
TIMESTEPS        = 1000
SCHEDULE         = 'linear'
DEVICE           = 'cuda'

# Model type for the DDPM arm: 'diffusion' (nb08) or 'ldm' (ablation)
MODEL2   = 'diffusion'
VAE_DIR  = None  # only needed when MODEL2='ldm'

# Ablation grids
FM_EULER_STEPS = [5, 10, 25, 50, 100]   # NFE = euler steps
DDIM_SKIPS     = [50, 20, 10, 5, 1]     # NFE = TIMESTEPS / skip

# Analysis scope
BATCH_SIZE     = 4
N_ABL_BATCHES  = 4    # batches from test set; increase for more stable estimates
ANALYSIS_CH    = 0    # temperature channel
T_LIQ          = 1700.0
LIQ_THR        = 0.5

# Output
EVAL_OUT_DIR = '/trace/group/forgelab/ngng/multifield/eval_results/fm_vs_ddpm_steps_study'

In [ ]:
%matplotlib inline
import os, sys, time
from pathlib import Path
import numpy as np, torch, pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import display as _ipy_display

plt.show = lambda *a, **kw: [_ipy_display(plt.figure(n)) for n in plt.get_fignums()] or plt.close('all')
if not torch.cuda.is_available() and DEVICE == 'cuda':
    print('No GPU — falling back to CPU (will be slow)'); DEVICE = 'cpu'

def find_root(s=Path.cwd()):
    for p in [s, *s.parents]:
        if (p / 'setup.py').exists() and (p / 'diffusionsr').exists(): return p
    raise RuntimeError('project root not found')
PROJECT_ROOT = find_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT: {PROJECT_ROOT} | device: {DEVICE}')

In [ ]:
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.analysis.analysis_functions import get_profile

def as_numpy(x): return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)
def mae(p, g):   return float(np.mean(np.abs(np.asarray(p).ravel() - np.asarray(g).ravel())))

kw = dict(downscale_method=DOWNSCALE_METHOD, root_folder=
          # infer DATA_ROOT from project layout
          str(PROJECT_ROOT / 'data' / 'data_fields'),
          normalize=NORMALIZE, n_steps=N_STEPS, field_names=FIELD_NAMES)
# Override DATA_ROOT if the default doesn't exist
_data_root = PROJECT_ROOT / 'data' / 'data_fields'
if not _data_root.exists():
    _data_root = Path('/trace/group/forgelab/ngng/multifield/data_fields')
kw['root_folder'] = str(_data_root)
train_ds, dev_ds, test_ds = (SimulationXZDataset(split=s, **kw) for s in ['train', 'dev', 'test'])
print(f'Fields: {train_ds.field_names}  HR: {train_ds.img_shape}  factor: {train_ds.factor}x')

In [ ]:
from diffusionsr.runners.train_flow_matching import FlowMatchingModel

fm_model = FlowMatchingModel(
    results_folder=FM_RUN_DIR, lr_encoder_folder=FM_ENC_DIR,
    train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=test_ds,
    timesteps=TIMESTEPS, conditioning='implicit', encoding=True,
    schedule=SCHEDULE, device=DEVICE, enc_output=False,
)
fm_model.load_saved_model()
print(f'FlowMatchingModel loaded from {FM_RUN_DIR}')

In [ ]:
if MODEL2 == 'ldm':
    from diffusionsr.runners.train_ldm import LDMModel
    diff_model = LDMModel(
        vae_folder=VAE_DIR, results_folder=DIFF_RUN_DIR, lr_encoder_folder=DIFF_ENC_DIR,
        train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=test_ds,
        timesteps=TIMESTEPS, conditioning='implicit', encoding=True,
        schedule=SCHEDULE, device=DEVICE, enc_output=False,
    )
else:  # 'diffusion'
    from diffusionsr.runners.train_diffusion import DiffusionModel
    diff_model = DiffusionModel(
        results_folder=DIFF_RUN_DIR, lr_encoder_folder=DIFF_ENC_DIR,
        train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=test_ds,
        timesteps=TIMESTEPS, conditioning='implicit', encoding=True,
        schedule=SCHEDULE, device=DEVICE, enc_output=False,
    )
diff_model.load_saved_model()
print(f'{MODEL2.upper()} loaded from {DIFF_RUN_DIR}')

In [ ]:
# ── Run step ablations on both models ─────────────────────────────────────────
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

fm_mae_d   = {n: [] for n in FM_EULER_STEPS}
fm_mp_d    = {n: [] for n in FM_EULER_STEPS}
fm_time_d  = {n: [] for n in FM_EULER_STEPS}

dd_mae_d   = {s: [] for s in DDIM_SKIPS}
dd_mp_d    = {s: [] for s in DDIM_SKIPS}
dd_time_d  = {s: [] for s in DDIM_SKIPS}

for batch_i, batch in enumerate(test_dl):
    if batch_i >= N_ABL_BATCHES: break
    _, hr_b, lr_b, ul_b = batch[:4]

    fm_xe   = fm_model.compute_x_e(lr_b, ul_b)
    diff_xe = diff_model.compute_x_e(lr_b, ul_b)

    for ns in FM_EULER_STEPS:
        t0 = time.perf_counter()
        with torch.no_grad():
            _s = fm_model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE),
                                       x_e=fm_xe, sampler='euler', n_steps=ns)
        fm_time_d[ns].append((time.perf_counter() - t0) / hr_b.shape[0])
        for si in range(hr_b.shape[0]):
            p = test_ds.unscale_data(_s[-1].cpu().numpy()[si], input_type='hr')
            g = test_ds.unscale_data(as_numpy(hr_b[si]),       input_type='hr')
            fm_mae_d[ns].append(mae(p[ANALYSIS_CH], g[ANALYSIS_CH]))
            try:
                pmp, _ = get_profile(p[ANALYSIS_CH:ANALYSIS_CH+1])
                gmp, _ = get_profile(g[ANALYSIS_CH:ANALYSIS_CH+1])
                fm_mp_d[ns].append(float(np.mean(np.abs(pmp - gmp))))
            except Exception: pass

    for sk in DDIM_SKIPS:
        t0 = time.perf_counter()
        with torch.no_grad():
            _s = diff_model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE),
                                         x_e=diff_xe, sampler='DDIM', skip=sk)
        dd_time_d[sk].append((time.perf_counter() - t0) / hr_b.shape[0])
        for si in range(hr_b.shape[0]):
            p = test_ds.unscale_data(_s[-1].cpu().numpy()[si], input_type='hr')
            g = test_ds.unscale_data(as_numpy(hr_b[si]),       input_type='hr')
            dd_mae_d[sk].append(mae(p[ANALYSIS_CH], g[ANALYSIS_CH]))
            try:
                pmp, _ = get_profile(p[ANALYSIS_CH:ANALYSIS_CH+1])
                gmp, _ = get_profile(g[ANALYSIS_CH:ANALYSIS_CH+1])
                dd_mp_d[sk].append(float(np.mean(np.abs(pmp - gmp))))
            except Exception: pass

print('Ablation complete.')
for ns in FM_EULER_STEPS:
    print(f'  FM  Euler n={ns:3d}  MAE={np.nanmean(fm_mae_d[ns]):.4f}  MP-MAE={np.nanmean(fm_mp_d[ns]) if fm_mp_d[ns] else float("nan"):.2f}px  {np.nanmean(fm_time_d[ns]):.2f}s/sample')
for sk in DDIM_SKIPS:
    nfe = TIMESTEPS // sk
    print(f'  {MODEL2.upper()} DDIM skip={sk:2d} (NFE={nfe:4d})  MAE={np.nanmean(dd_mae_d[sk]):.4f}  MP-MAE={np.nanmean(dd_mp_d[sk]) if dd_mp_d[sk] else float("nan"):.2f}px  {np.nanmean(dd_time_d[sk]):.2f}s/sample')

In [ ]:
# ── Quality vs. NFE curves ─────────────────────────────────────────────────────
fm_nfe   = FM_EULER_STEPS
dd_nfe   = [TIMESTEPS // sk for sk in DDIM_SKIPS]

fm_mae_v  = [np.nanmean(fm_mae_d[n])  for n in FM_EULER_STEPS]
fm_mp_v   = [np.nanmean(fm_mp_d[n])   if fm_mp_d[n]  else float('nan') for n in FM_EULER_STEPS]
fm_time_v = [np.nanmean(fm_time_d[n]) for n in FM_EULER_STEPS]

dd_mae_v  = [np.nanmean(dd_mae_d[s])  for s in DDIM_SKIPS]
dd_mp_v   = [np.nanmean(dd_mp_d[s])   if dd_mp_d[s]  else float('nan') for s in DDIM_SKIPS]
dd_time_v = [np.nanmean(dd_time_d[s]) for s in DDIM_SKIPS]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=140)
for ax, fm_y, dd_y, ylabel in [
        (axes[0], fm_mae_v, dd_mae_v, 'Field MAE (temperature)'),
        (axes[1], fm_mp_v,  dd_mp_v,  'MP-MAE (px)')]:
    ax.plot(fm_nfe, fm_y, 'o-',  color='darkorange', lw=2.5, ms=9, label='FlowMatching (Euler)')
    ax.plot(dd_nfe, dd_y, 's--', color='steelblue',  lw=2.5, ms=9, label=f'{MODEL2.upper()} (DDIM)')
    # annotate each point with its step count
    for i, ns in enumerate(fm_nfe):
        ax.annotate(str(ns), (fm_nfe[i], fm_y[i]), fontsize=8,
                    color='darkorange', xytext=(4, 4), textcoords='offset points')
    for i, sk in enumerate(DDIM_SKIPS):
        ax.annotate(f'{dd_nfe[i]}', (dd_nfe[i], dd_y[i]), fontsize=8,
                    color='steelblue', xytext=(4, -10), textcoords='offset points')
    ax.set_xlabel('NFE (number of function evaluations)')
    ax.set_ylabel(ylabel); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
axes[0].set_title('Field MAE vs. NFE')
axes[1].set_title('Melt-Pool Depth Error vs. NFE')
plt.suptitle(
    f'FlowMatching (Euler) vs {MODEL2.upper()} (DDIM) — fields={FIELD_NAMES}\n'
    f'{N_ABL_BATCHES} test batches × {BATCH_SIZE} samples',
    fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── Quality vs. wall-clock time (Pareto scatter) ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=140)
for ax, fm_y, dd_y, ylabel in [
        (axes[0], fm_mae_v, dd_mae_v, 'Field MAE'),
        (axes[1], fm_mp_v,  dd_mp_v,  'MP-MAE (px)')]:
    ax.scatter(fm_time_v, fm_y, c=fm_nfe,  cmap='Oranges', s=130, zorder=5,
               edgecolors='darkorange', linewidths=1.5, label='FlowMatching (Euler)')
    ax.scatter(dd_time_v, dd_y, c=dd_nfe,  cmap='Blues',   s=130, zorder=5,
               edgecolors='steelblue',  linewidths=1.5, marker='s', label=f'{MODEL2.upper()} (DDIM)')
    ax.plot(fm_time_v, fm_y, '--', color='darkorange', lw=1.2, alpha=0.5)
    ax.plot(dd_time_v, dd_y, '--', color='steelblue',  lw=1.2, alpha=0.5)
    for i, ns in enumerate(fm_nfe):
        ax.annotate(f'{ns}', (fm_time_v[i], fm_y[i]), fontsize=8,
                    color='darkorange', xytext=(4, 3), textcoords='offset points')
    for i, sk in enumerate(DDIM_SKIPS):
        ax.annotate(f'{dd_nfe[i]}', (dd_time_v[i], dd_y[i]), fontsize=8,
                    color='steelblue', xytext=(4, -10), textcoords='offset points')
    ax.set_xlabel('Inference time per sample (s)')
    ax.set_ylabel(ylabel); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
axes[0].set_title('Field MAE vs. Time')
axes[1].set_title('MP-MAE vs. Time')
plt.suptitle(
    'Quality vs. Inference Cost — FM Euler (orange) vs DDPM DDIM (blue)\n'
    'Numbers = NFE. Lower-left = better.',
    fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
rows = []
for ns in FM_EULER_STEPS:
    rows.append({
        'model': 'FlowMatching', 'sampler': f'Euler n={ns}', 'nfe': ns,
        'time_s':  float(np.nanmean(fm_time_d[ns])),
        'mae':     float(np.nanmean(fm_mae_d[ns])),
        'mp_mae':  float(np.nanmean(fm_mp_d[ns])) if fm_mp_d[ns] else float('nan'),
    })
for sk in DDIM_SKIPS:
    rows.append({
        'model': MODEL2, 'sampler': f'DDIM skip={sk}', 'nfe': TIMESTEPS // sk,
        'time_s':  float(np.nanmean(dd_time_d[sk])),
        'mae':     float(np.nanmean(dd_mae_d[sk])),
        'mp_mae':  float(np.nanmean(dd_mp_d[sk])) if dd_mp_d[sk] else float('nan'),
    })
df = pd.DataFrame(rows).sort_values('mae')
print('Results sorted by MAE (best first):')
print(df.to_string(index=False, float_format='{:.4f}'.format))

os.makedirs(EVAL_OUT_DIR, exist_ok=True)
df.to_csv(f'{EVAL_OUT_DIR}/fm_vs_ddpm_steps_study.csv', index=False)
print(f'\nSaved -> {EVAL_OUT_DIR}/fm_vs_ddpm_steps_study.csv')